# Phase 4 Step 3 — Item-CF
Controlled validation-only personalization experiment. **Lockbox recommendation performance is not inspected.**

## Objective and frozen contract
Test whether binary implicit Item-CF improves next-interaction full-catalog ranking over frozen popularity. History is through `T`; training similarities use only pre-validation events; seen items and cold-target misses remain.

In [ ]:
from pathlib import Path
import json, pandas as pd
from marketmind.recommendations.item_cf_experiment import run_experiment
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
ARTIFACTS = ROOT/'reports/recommendation/artifacts'


## Training interaction matrix and item similarity
Repeated visitor-item events collapse to one. Exact sparse cosine co-occurrence retains at most 200 neighbors; no dense 211,905² matrix is created. Self-similarity is retained because repeat recommendations are allowed.

In [ ]:
# Reproducible runner discards lockbox rows before training or evaluation.
summary = run_experiment(ROOT/'data/raw/retailrocket/events.csv', ARTIFACTS)
pd.Series(summary['matrix'])


## Point-in-time semantics and configurations
Similarities are frozen from events before 2015-08-17 03:00 UTC. Configurations 50/100/200 were predeclared; macro NDCG@10 selects one.

In [ ]:
pd.DataFrame(summary['configuration_results'])[['neighbors','ndcg_at_10','hit_rate_at_10','hit_rate_at_20','coverage_at_20']]


## Full-catalog results and popularity comparison

In [ ]:
display(pd.Series(summary['overall_metrics']))
display(pd.Series(summary['comparison']))


## Repeat/novel, target-event, history-depth, and cold-target slices

In [ ]:
slices = pd.read_csv(ARTIFACTS/'item_cf_slice_results.csv')
slices[['dimension','value','instances','ndcg_at_10','hit_rate_at_10','hit_rate_at_20']]


## Personalization diagnostics, fallback, coverage, and compute profile

In [ ]:
display(pd.Series(summary['personalization']))
display(pd.DataFrame(summary['coverage']).T)
display(pd.Series(summary['compute']))


## Deterministic error analysis
The first five visitor IDs in each popularity/CF hit-miss category are retained, avoiding cherry-picking.

In [ ]:
pd.read_csv(ARTIFACTS/'item_cf_error_sample.csv')


## Decision
Item-CF is clearly justified on validation, but repeat/self-similarity drives much of the gain and temporal stability is unknown. Novel-target ranking remains weak. One controlled latent-factor challenger is justified; no final model is selected and lockbox rankings remain untouched.